# Bank Marketing Data Exploration

This notebook loads, explores, cleans and analyses the Bank Marketing dataset.

## 1. Import libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## 2. Load the raw dataset

In [ ]:
raw_data_path = Path("../data/raw/bank_marketing_raw.csv")
clean_data_path = Path("../data/clean/bank_marketing_cleaned.csv")

df = pd.read_csv(raw_data_path)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")

In [ ]:
df.head()

## 3. Initial data exploration

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Class"].value_counts()

### Initial observations

- The dataset contains 45,211 customer records and 17 columns.
- No null values or duplicate rows were identified.
- The original column names are not descriptive.
- Several categorical fields use `"unknown"` to represent unavailable information.

## 4. Data cleaning

In [ ]:
df.columns = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "balance",
    "housing",
    "loan",
    "contact",
    "day",
    "month",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
    "subscribed",
]

In [ ]:
df["subscribed"] = df["subscribed"].replace({
    1: "No",
    2: "Yes",
})

In [ ]:
df["subscribed"].value_counts()

In [ ]:
(df == "unknown").sum()

### Data cleaning summary

- Renamed all columns using meaningful names.
- Converted the target values from `1` and `2` to `"No"` and `"Yes"`.
- Retained `"unknown"` values because they represent unavailable information rather than null values.
- Saved the cleaned dataset for use in SQL and Tableau.

In [ ]:
clean_data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(clean_data_path, index=False)
print(f"Cleaned dataset saved to: {clean_data_path}")

## 5. Exploratory data analysis

### Subscription overview

In [ ]:
df["subscribed"].value_counts()

In [ ]:
df["subscribed"].value_counts(normalize=True).mul(100).round(2)

### Customer age

In [ ]:
df["age"].describe()

### Customer distribution by occupation

In [ ]:
job_counts = df["job"].value_counts().sort_values()

job_counts.plot(kind="barh", figsize=(10, 6))
plt.title("Number of Customers by Job")
plt.xlabel("Number of Customers")
plt.ylabel("Job")
plt.tight_layout()
plt.show()

**Insight:** Blue-collar workers represent the largest customer segment, followed by management and technicians.

### Subscription rate by occupation

In [ ]:
job_subscription = (
    pd.crosstab(df["job"], df["subscribed"], normalize="index")
    .mul(100)
    .round(2)
)

job_subscription

In [ ]:
job_subscription["Yes"].sort_values().plot(
    kind="barh",
    figsize=(10, 6),
)

plt.title("Subscription Rate by Occupation")
plt.xlabel("Subscription Rate (%)")
plt.ylabel("Occupation")
plt.tight_layout()
plt.show()

### Subscription rate by month

In [ ]:
month_subscription = (
    pd.crosstab(df["month"], df["subscribed"], normalize="index")
    .mul(100)
    .round(2)
)

month_subscription

In [ ]:
month_subscription["Yes"].sort_values().plot(
    kind="barh",
    figsize=(10, 6),
)

plt.title("Subscription Rate by Month")
plt.xlabel("Subscription Rate (%)")
plt.ylabel("Month")
plt.tight_layout()
plt.show()

**Insight:** March recorded the highest subscription rate, while May recorded the lowest. Campaign timing may therefore influence customer response.

### Subscription rate by housing-loan status

In [ ]:
housing_subscription = (
    pd.crosstab(df["housing"], df["subscribed"], normalize="index")
    .mul(100)
    .round(2)
)

housing_subscription

In [ ]:
housing_subscription["Yes"].plot(kind="bar", figsize=(6, 4))

plt.title("Subscription Rate by Housing Loan")
plt.xlabel("Housing Loan")
plt.ylabel("Subscription Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Customers without a housing loan subscribed at a higher rate than customers with a housing loan.

### Average age by subscription status

In [ ]:
df.groupby("subscribed")["age"].mean().round(2)

In [ ]:
df.groupby("subscribed")["age"].mean().plot(
    kind="bar",
    figsize=(6, 4),
)

plt.title("Average Age by Subscription Status")
plt.xlabel("Subscription")
plt.ylabel("Average Age")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Customers who subscribed were slightly older on average, although the difference was relatively small.